In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
import pandas as pd

In [3]:
class MLP(nn.Module):
    def __init__(self, input_size):
        super(MLP, self).__init__()
        #структура нейросети: input_size - 128 - 64 - 32 - 1
        self.layers = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.layers(x)

In [4]:
average_bill = pd.read_csv('../../data/df_train_average_bill_encoded_scaled.csv')

In [5]:
#делим на 60000, потому что остальные переменные в районе -1 и 1, веса становятся слишком большими и результаты становятся нестабильными
average_bill['Средний чек'] = average_bill['Средний чек'] / 1000
average_bill

,Средний чек,Численность населения,Количество домохозяйств,"Трафик пеший, в час","Трафик авто, в час",month_sin,month_cos,pca_1,pca_2,pca_3,pca_4,Населенный пункт_fold_average_bill,Регион_fold_average_bill,"Дата открытия, категориальный_Новый","Дата открытия, категориальный_Открыт давно","Дата открытия, категориальный_Средний по возрасту","Торговая площадь, категориальный_Большой","Торговая площадь, категориальный_Маленький","Торговая площадь, категориальный_Очень большой","Торговая площадь, категориальный_Средний"
0,0.823060,-0.183291,-0.634931,-0.658506,0.29525,-8.660254e-01,5.000000e-01,-0.647364,0.219085,0.058661,-1.248180,0.087335,-0.086181,0,0,1,0,0,0,1
1,0.859362,-0.183291,-0.634931,-0.658506,0.29525,5.000000e-01,-8.660254e-01,-0.647364,0.219085,0.058661,-1.248180,0.087335,-0.086181,0,0,1,0,0,0,1
2,0.763938,-0.183291,-0.634931,-0.658506,0.29525,5.000000e-01,8.660254e-01,-0.647364,0.219085,0.058661,-1.248180,0.087335,-0.086181,0,0,1,0,0,0,1
3,0.836362,-0.183291,-0.634931,-0.658506,0.29525,1.224647e-16,-1.000000e+00,-0.647364,0.219085,0.058661,-1.248180,0.087335,-0.086181,0,0,1,0,0,0,1
4,0.845258,-0.183291,-0.634931,-0.658506,0.29525,-5.000000e-01,-8.660254e-01,-0.647364,0.219085,0.058661,-1.248180,0.087335,-0.086181,0,0,1,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
232664,1.167101,-0.187406,-0.712807,-0.009117,0.52820,-8.660254e-01,5.000000e-01,-1.678946,0.112034,-0.095371,-0.163764,-0.387657,-0.464348,1,0,0,0,0,0,1
232665,1.252914,-0.187406,-0.712807,-0.009117,0.52820,-5.000000e-01,8.660254e-01,-1.678946,0.112034,-0.095371,-0.163764,-0.387657,-0.464348,1,0,0,0,0,0,1
232666,1.130824,-0.187406,-0.712807,-0.009117,0.52820,-1.000000e+00,-1.836970e-16,-1.678946,0.112034,-0.095371,-0.163764,-0.387657,-0.464348,1,0,0,0,0,0,1
232667,1.461929,-0.187406,-0.712807,-0.009117,0.52820,-2.449294e-16,1.000000e+00,-1.678946,0.112034,-0.095371,-0.163764,-0.387657,-0.464348,1,0,0,0,0,0,1


In [6]:
mlp = MLP(input_size=(average_bill.shape[1] - 1))
#функци потерь, сочетающая RMSE и MAE
criterion = nn.HuberLoss(delta=1.0)
optimizer = optim.Adam(mlp.parameters(), lr=0.001)

In [7]:
y_average_bill = average_bill['Средний чек']
x_average_bill = average_bill.drop(['Средний чек'], axis=1)

In [8]:
from sklearn.model_selection import train_test_split

x_train_average_bill, x_test_average_bill, y_train_average_bill, y_test_average_bill = train_test_split(x_average_bill, y_average_bill, test_size=0.2, random_state=598)

In [9]:
#pytorch требует данных в формате тензоров
x_tensor = torch.tensor(x_train_average_bill.values, dtype=torch.float32)
y_tensor = torch.tensor(y_train_average_bill.values, dtype=torch.float32).reshape(-1, 1)

In [10]:
from torch.utils.data import DataLoader, TensorDataset
dataset = TensorDataset(x_tensor, y_tensor)
loader = DataLoader(dataset, batch_size=100, shuffle=True)

In [11]:
from torch.nn.functional import l1_loss

In [12]:
x_test_tensor = torch.tensor(x_test_average_bill.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_average_bill.values, dtype=torch.float32).reshape(-1, 1)

In [13]:
for epoch in range(300):
    mlp.train()
    for batch_x, batch_y in loader:
        optimizer.zero_grad()
        
        predictions = mlp(batch_x)
        loss = criterion(predictions, batch_y)
        
        loss.backward()
        optimizer.step()

    mlp.eval()    
    with torch.no_grad():
        train_preds = mlp(x_tensor)
        rmse_train = torch.sqrt(torch.mean((train_preds - y_tensor) ** 2)).item() * 1000
        mae_train = l1_loss(train_preds * 1000, y_tensor * 1000).item()
        
        test_preds = mlp(x_test_tensor)
        rmse_test = torch.sqrt(torch.mean((test_preds - y_test_tensor) ** 2)).item() * 1000
        mae_test = l1_loss(test_preds * 1000, y_test_tensor * 1000).item()
        
    print(f"Эпоха {epoch}"
          f" Train RMSE: {rmse_train:7.2f}, MAE: {mae_train:7.2f}"
          f" Test RMSE: {rmse_test:7.2f}, MAE: {mae_test:7.2f}")

Эпоха 0 Train RMSE:  203.56, MAE:  153.51 Test RMSE:  204.55, MAE:  154.22
Эпоха 1 Train RMSE:  200.50, MAE:  149.21 Test RMSE:  202.07, MAE:  150.47
Эпоха 2 Train RMSE:  198.47, MAE:  147.92 Test RMSE:  200.13, MAE:  149.17
Эпоха 3 Train RMSE:  194.08, MAE:  146.99 Test RMSE:  195.84, MAE:  148.20
Эпоха 4 Train RMSE:  196.60, MAE:  150.59 Test RMSE:  198.98, MAE:  152.04
Эпоха 5 Train RMSE:  189.86, MAE:  143.12 Test RMSE:  192.66, MAE:  144.92
Эпоха 6 Train RMSE:  192.44, MAE:  147.39 Test RMSE:  195.05, MAE:  148.97
Эпоха 7 Train RMSE:  188.16, MAE:  141.33 Test RMSE:  191.06, MAE:  143.57
Эпоха 8 Train RMSE:  182.56, MAE:  137.98 Test RMSE:  186.02, MAE:  140.42
Эпоха 9 Train RMSE:  180.02, MAE:  137.10 Test RMSE:  183.60, MAE:  139.58
Эпоха 10 Train RMSE:  177.84, MAE:  134.29 Test RMSE:  181.89, MAE:  137.21
Эпоха 11 Train RMSE:  174.14, MAE:  131.74 Test RMSE:  178.41, MAE:  134.85
Эпоха 12 Train RMSE:  172.38, MAE:  131.00 Test RMSE:  176.80, MAE:  134.04
Эпоха 13 Train RMSE:  